# WP4 — Structure Prediction for the Top-K Candidates

We fold only the K diversity-selected candidates from WP3 (cheap to expand
later via `ranking.pre_fold_top_k`). Benchmark structures are pulled from
AlphaFold DB (free, instant). Ancestral candidates are folded via ColabFold
AF2 or ESMFold depending on `structure.folder_provider`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from asr_poc.config import load_config
from asr_poc.io_utils import set_seed
cfg = load_config(ROOT / "config" / "target.yaml")
set_seed(cfg.run.seed)
cfg.paths.ensure_dirs()
print("Target:", cfg.target.name, "| embeddings:", cfg.embeddings.provider, "| report.top_n:", cfg.report.top_n)

Target: lipase | embeddings: local | report.top_n: 3


## 1. Read the diversity-selected fold list from WP3

In [2]:
import pandas as pd
from asr_poc import ranking, structure
signals = pd.read_csv(cfg.paths.signals_csv) if cfg.paths.signals_csv.exists() else None
if signals is None:
    signals = ranking.pre_fold_rank(cfg)
    signals = signals.reset_index()
fold_ids = signals.head(cfg.ranking.pre_fold_top_k)["candidate_id"].tolist()
print("Folding:", fold_ids)

Folding: ['ALT_Node52_alt2', 'ALT_Node4_alt1', 'ALT_Node47_alt3']


## 2. Predict structures + compute metrics

In [3]:
metrics = structure.analyze_candidates(cfg, ranking=pd.DataFrame({"candidate_id": fold_ids}))
metrics

09:19:42 [info     ] wp4_complete                   n=8 out=C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project\reports\structure_metrics.csv stage=wp4.structure


,candidate_id,kind,pdb,radius_of_gyration,mean_plddt,contact_density
0,ALT_Node52_alt2,candidate,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,1931.756990,60.053515,1.998296
1,ALT_Node4_alt1,candidate,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,2134.695540,59.746567,1.998458
2,ALT_Node47_alt3,candidate,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,1904.332862,59.853324,1.998272
3,BENCH_A2VBC4,benchmark,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,19.230403,92.030807,5.118012
4,BENCH_A8PUY1,benchmark,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,21.418318,90.224671,5.006579
5,BENCH_F4JFU8,benchmark,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,23.712226,84.168050,4.777992
6,BENCH_O00748,benchmark,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,26.027093,91.019678,5.048301
7,BENCH_O22527,benchmark,C:\Users\gcv\Downloads\AI_POC 2\AI_POC\project...,19.467700,91.787870,5.129630
